# L6: Lab - Phone Photo Memory

In this lab, you'll build a personal photo memory app for an Android phone. Over the course of a week, your phone captures dozens of photos. CLIP (compiled for Snapdragon via AI Hub) embeds each photo, and Qdrant Edge stores everything locally on-device. Later, you search your photos with plain English: "what did I eat on Tuesday?" or "show me that whiteboard from the meeting."

No cloud upload. No privacy concerns. Everything runs on your phone's Snapdragon chip.

```
Phone Camera (5 days, 70+ photos)
     |
  CLIP (Snapdragon NPU)
     |
  Qdrant Edge (on-device storage)
     |
  "What did I eat on Tuesday?"
     |
  Matching photos + metadata
```

## Setup

In [ ]:
!pip install qdrant-edge-py qai-hub "qai-hub-models[openai_clip]" torch Pillow matplotlib numpy

In [ ]:
import os
import sys
sys.path.append("..")

import torch
import clip
import numpy as np
import time
import datetime
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
from qai_hub_models.models.openai_clip import Model as OpenAIClip
from qdrant_edge import (
    EdgeShard, EdgeConfig, VectorDataConfig, Distance,
    Point, UpdateOperation, Query, QueryRequest,
)

# Load CLIP from qai_hub_models
clip_model = OpenAIClip.from_pretrained()

VECTOR_NAME = "photo"
DIMENSION = 512

print("CLIP model loaded from qai_hub_models")

### Compile CLIP for Your Phone

Compile the visual encoder so it runs on your phone's Snapdragon NPU.

In [ ]:
import qai_hub
from utils import get_ai_hub_api_token, get_random_device

ai_hub_api_token = get_ai_hub_api_token()
!qai-hub configure --api_token $ai_hub_api_token

device_name = get_random_device()
device = qai_hub.Device(device_name)

# Wrap and trace the visual encoder
class CLIPVisualEncoder(torch.nn.Module):
    def __init__(self, clip_visual):
        super().__init__()
        self.visual = clip_visual

    def forward(self, image):
        features = self.visual(image)
        return features / features.norm(dim=1, keepdim=True)

visual_encoder = CLIPVisualEncoder(clip_model.clip.visual)
example_image = torch.randn(1, 3, 224, 224)
traced_visual = torch.jit.trace(visual_encoder, example_image)

compile_job = qai_hub.submit_compile_job(
    model=traced_visual,
    input_specs={"image": (1, 3, 224, 224)},
    device=device,
)
target_model = compile_job.get_target_model()

profile_job = qai_hub.submit_profile_job(model=target_model, device=device)
profile = profile_job.download_profile()
print(f"CLIP compiled for {device_name}")
print(f"Inference time: {profile['execution_summary']['estimated_inference_time']}")

## 1. Simulate a Week of Phone Photos

We'll generate synthetic photos your phone captures over 5 days (Monday through Friday). Each day has a different routine with 12-16 moments: commutes, meetings, meals, outdoor time, and more.

In [ ]:
PHOTOS_DIR = "./phone_photos"
Path(PHOTOS_DIR).mkdir(parents=True, exist_ok=True)

np.random.seed(42)

def make_photo(name, base_color, noise_level=20):
    """Generate a synthetic photo with color noise."""
    base = np.array(base_color, dtype=np.uint8)
    noise = np.random.randint(-noise_level, noise_level, (224, 224, 3), dtype=np.int16)
    frame = np.clip(base + noise, 0, 255).astype(np.uint8)
    img = Image.fromarray(frame)
    path = f"{PHOTOS_DIR}/{name}.png"
    img.save(path)
    return path

# Photo templates: (activity, base_color, location)
# Colors are approximate visual representations
photo_templates = {
    "morning": [
        ("Morning coffee at the corner cafe", [139, 90, 43], "cafe"),
        ("Sunrise view from the apartment window", [220, 160, 100], "home"),
        ("Breakfast: eggs and toast", [200, 170, 80], "home"),
        ("Smoothie bowl with berries", [150, 60, 90], "home"),
    ],
    "commute": [
        ("On the train, reading the news", [100, 100, 120], "transit"),
        ("Walking past the mural on 5th street", [160, 80, 120], "outdoors"),
        ("Bus stop in the rain", [90, 95, 110], "transit"),
        ("Bike ride through downtown", [130, 150, 130], "outdoors"),
    ],
    "office": [
        ("Settling in at the desk", [200, 200, 210], "office"),
        ("Morning standup with the team", [180, 190, 200], "office"),
        ("Architecture diagram on the whiteboard", [240, 240, 245], "office"),
        ("Deep work: coding the sync module", [40, 42, 54], "office"),
        ("Design review for the context hub", [170, 180, 200], "office"),
        ("Pair programming on search optimization", [50, 50, 65], "office"),
        ("Whiteboard brainstorm session", [235, 235, 240], "office"),
        ("Conference room setup for demo", [190, 195, 205], "office"),
    ],
    "lunch": [
        ("Fish tacos at the place on 3rd", [180, 120, 60], "restaurant"),
        ("Sushi platter from the new spot", [200, 150, 130], "restaurant"),
        ("Burrito bowl at Chipotle", [160, 100, 50], "restaurant"),
        ("Salad from the farmers market", [80, 160, 60], "outdoors"),
        ("Pizza slice from the cart outside", [200, 140, 70], "outdoors"),
        ("Pho from the Vietnamese place", [180, 160, 130], "restaurant"),
    ],
    "afternoon": [
        ("Walking through the park after lunch", [80, 160, 80], "outdoors"),
        ("Nice sunset on the walk home", [220, 140, 80], "outdoors"),
        ("Dog spotted in the park", [140, 130, 90], "outdoors"),
        ("Street performer near the office", [160, 140, 120], "outdoors"),
        ("Flowers blooming at the plaza", [180, 100, 140], "outdoors"),
    ],
    "errand": [
        ("Picking up ingredients for dinner", [200, 210, 180], "store"),
        ("Browsing at the bookstore", [120, 90, 60], "store"),
        ("Package pickup at the post office", [180, 180, 175], "store"),
        ("New shoes at the outlet", [210, 200, 190], "store"),
    ],
    "evening": [
        ("Cooking pasta with fresh vegetables", [220, 180, 100], "home"),
        ("Grilled chicken with roasted potatoes", [180, 140, 70], "home"),
        ("Stir fry with tofu and broccoli", [100, 150, 70], "home"),
        ("Reading on the couch", [60, 50, 40], "home"),
        ("Movie night setup", [30, 30, 45], "home"),
        ("Evening yoga session", [180, 170, 160], "home"),
        ("View from the balcony at night", [20, 25, 50], "home"),
    ],
}

# Generate 5 days of photos with different daily routines
day_names = ["monday", "tuesday", "wednesday", "thursday", "friday"]
day_schedules = {
    "monday":    ["morning", "commute", "office", "office", "lunch", "office", "office", "afternoon", "commute", "errand", "evening", "evening"],
    "tuesday":   ["morning", "commute", "office", "office", "office", "lunch", "afternoon", "office", "commute", "evening", "evening", "evening"],
    "wednesday": ["morning", "morning", "commute", "office", "office", "lunch", "lunch", "office", "afternoon", "afternoon", "commute", "evening", "evening"],
    "thursday":  ["morning", "commute", "office", "office", "office", "lunch", "office", "office", "afternoon", "errand", "errand", "evening", "evening"],
    "friday":    ["morning", "commute", "office", "lunch", "afternoon", "afternoon", "office", "commute", "errand", "evening", "evening", "evening", "evening"],
}

photos = []
photo_idx = 0

for day_name in day_names:
    schedule = day_schedules[day_name]
    start_hour = 7
    for slot_idx, slot in enumerate(schedule):
        templates = photo_templates[slot]
        template = templates[np.random.randint(0, len(templates))]
        activity, color, location = template

        hour = min(start_hour + slot_idx, 22)
        minute = np.random.choice([0, 15, 30, 45])
        time_str = f"{hour:02d}:{minute:02d}"

        name = f"{day_name}_{photo_idx:03d}_{slot}"
        path = make_photo(name, color)

        photos.append({
            "path": path,
            "activity": activity,
            "location": location,
            "time_str": time_str,
            "day": day_name,
            "hour": hour,
            "slot": slot,
        })
        photo_idx += 1

print(f"Captured {len(photos)} photos across {len(day_names)} days")
from collections import Counter
day_counts = Counter(p["day"] for p in photos)
loc_counts = Counter(p["location"] for p in photos)
print(f"By day:      {dict(sorted(day_counts.items(), key=lambda x: day_names.index(x[0])))}")
print(f"By location: {dict(loc_counts)}")

# Preview: 2 photos per day
fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for col, day in enumerate(day_names):
    day_photos = [p for p in photos if p["day"] == day]
    for row in range(2):
        idx = row * (len(day_photos) // 2)
        if idx < len(day_photos):
            p = day_photos[idx]
            axes[row, col].imshow(Image.open(p["path"]))
            axes[row, col].set_title(f"{p['time_str']} {p['location']}", fontsize=8)
        axes[row, col].axis("off")
    axes[0, col].set_xlabel(day.capitalize(), fontsize=10)
plt.suptitle("A Week in Phone Photos (sample)", fontsize=14)
plt.tight_layout()
plt.show()

## 2. Embed and Store Photos

Embed each photo with CLIP and store in Qdrant Edge with metadata: time, location, activity.

In [ ]:
def embed_image(image_path):
    """Generate a 512-d embedding for a photo using CLIP."""
    img = Image.open(image_path).convert("RGB")
    img_tensor = clip_model.image_preprocessor(img).unsqueeze(0)
    with torch.no_grad():
        features = clip_model.clip.encode_image(img_tensor)
        features = features / features.norm(dim=1, keepdim=True)
    return features.squeeze(0).float().numpy()

def embed_text(text):
    """Generate a 512-d embedding for a text query using CLIP."""
    tokens = clip.tokenize([text])
    with torch.no_grad():
        features = clip_model.clip.encode_text(tokens)
        features = features / features.norm(dim=1, keepdim=True)
    return features.squeeze(0).float().numpy()

In [ ]:
SHARD_DIR = "./photo_memory_shard"
Path(SHARD_DIR).mkdir(parents=True, exist_ok=True)

config = EdgeConfig(
    vector_data={
        VECTOR_NAME: VectorDataConfig(
            size=DIMENSION,
            distance=Distance.Cosine,
        )
    }
)

shard = EdgeShard(SHARD_DIR, config)

today = datetime.date.today()
# Map days to actual dates (this week)
day_offsets = {
    "monday": today - datetime.timedelta(days=today.weekday()),
    "tuesday": today - datetime.timedelta(days=today.weekday() - 1),
    "wednesday": today - datetime.timedelta(days=today.weekday() - 2),
    "thursday": today - datetime.timedelta(days=today.weekday() - 3),
    "friday": today - datetime.timedelta(days=today.weekday() - 4),
}

print(f"Embedding and storing {len(photos)} photos...")
points = []
for i, photo in enumerate(photos):
    emb = embed_image(photo["path"])
    day_date = day_offsets[photo["day"]]
    ts = datetime.datetime.combine(day_date, datetime.time(photo["hour"], 0)).timestamp()

    points.append(Point(
        id=i,
        vector={VECTOR_NAME: emb.tolist()},
        payload={
            "activity": photo["activity"],
            "location": photo["location"],
            "time_str": photo["time_str"],
            "day": photo["day"],
            "timestamp": ts,
            "image_path": photo["path"],
        }
    ))

shard.update(UpdateOperation.upsert_points(points))
print(f"Stored {len(points)} photos in on-device memory")

## 3. Search Your Photos with Natural Language

CLIP puts images and text in the same embedding space. Type a question, get matching photos.

In [ ]:
def search_photos(question, limit=3, payload_filter=None):
    """Search photo memory using a text query."""
    query_emb = embed_text(question)
    kwargs = {
        "query": Query.Nearest(query_emb.tolist(), using=VECTOR_NAME),
        "limit": limit,
        "with_vector": False,
        "with_payload": True,
    }
    if payload_filter:
        kwargs["filter"] = payload_filter
    return shard.query(QueryRequest(**kwargs))

queries = [
    "What did I eat this week?",
    "Show me the whiteboard",
    "When was I outside?",
    "Meetings and presentations",
    "What did I cook for dinner?",
    "Coffee and breakfast",
]

for q in queries:
    print(f"\nQ: {q}")
    results = search_photos(q, limit=5)
    for r in results:
        p = r.payload
        print(f"  [{r.score:.3f}] {p['day'][:3]} {p['time_str']} @ {p['location']} - {p['activity']}")

## 4. Filter by Location

"What happened at the office?" combines semantic search with a location filter.

In [ ]:
print("Office photos across the week:")
results = search_photos(
    "work and meetings",
    limit=8,
    payload_filter={"must": [{"key": "location", "match": {"value": "office"}}]}
)
for r in results:
    p = r.payload
    print(f"  {p['day'][:3]} {p['time_str']} - {p['activity']}")

## 5. Filter by Time and Day

"What happened Wednesday afternoon?" combines day and time filtering across the week.

In [ ]:
# Wednesday only
print("Wednesday photos:")
results = search_photos(
    "what happened",
    limit=5,
    payload_filter={
        "must": [{"key": "day", "match": {"value": "wednesday"}}]
    }
)
for r in results:
    p = r.payload
    print(f"  {p['time_str']} @ {p['location']} - {p['activity']}")

# Afternoon across all days (hours 12-18)
print("\nAfternoon photos (all days, 12pm-6pm):")
afternoon_timestamps = []
for day_name, day_date in day_offsets.items():
    afternoon_start = datetime.datetime.combine(day_date, datetime.time(12, 0)).timestamp()
    afternoon_end = datetime.datetime.combine(day_date, datetime.time(18, 0)).timestamp()

results = search_photos(
    "outdoor activities",
    limit=5,
    payload_filter={
        "must": [{"key": "location", "match": {"value": "outdoors"}}]
    }
)
for r in results:
    p = r.payload
    print(f"  {p['day'][:3]} {p['time_str']} - {p['activity']}")

## 6. Weekly Timeline

Generate a timeline of your entire week from stored photo memories.

In [ ]:
all_memories = shard.retrieve(
    point_ids=list(range(len(photos))),
    with_payload=True,
    with_vector=False,
)

sorted_memories = sorted(all_memories, key=lambda p: p.payload["timestamp"])

print("Your week:")
print("=" * 60)
current_day = None
current_location = None
for m in sorted_memories:
    p = m.payload
    if p["day"] != current_day:
        current_day = p["day"]
        current_location = None
        print(f"\n{'=' * 20} {current_day.upper()} {'=' * 20}")
    if p["location"] != current_location:
        current_location = p["location"]
        print(f"\n  [{current_location.upper()}]")
    print(f"  {p['time_str']}  {p['activity']}")

from collections import Counter
locations = Counter(m.payload["location"] for m in all_memories)
days = Counter(m.payload["day"] for m in all_memories)
print(f"\n{'=' * 60}")
print(f"Total photos: {len(all_memories)}")
print(f"Places visited: {dict(locations)}")
print(f"Photos per day: {dict(sorted(days.items(), key=lambda x: day_names.index(x[0])))}")

## 7. Cleanup

In [ ]:
shard.close()

import shutil
shutil.rmtree(SHARD_DIR, ignore_errors=True)
shutil.rmtree(PHOTOS_DIR, ignore_errors=True)
print("Cleaned up")

## Summary

In this lab you built a personal photo memory app:
- Compiled CLIP for a Snapdragon phone via AI Hub
- Captured and embedded 70+ photos across a 5-day work week
- Searched your photos with natural language ("What did I eat this week?", "Show me the whiteboard")
- Filtered by location, day, and time window
- Generated a weekly timeline across 6+ locations

Everything runs on-device. No cloud uploads, no privacy concerns, sub-20ms queries. This is the foundation of personal AI memory on your phone.

In the next lab, you'll build a product scout that helps you compare items across shopping trips.